In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from pytorch_lightning.callbacks import Callback
import pandas as pd
import pickle
from pytorch_lightning.callbacks import ModelCheckpoint

D:\Anaconda\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.cuda.get_device_name(0)

'NVIDIA GeForce RTX 3060 Laptop GPU'

In [66]:
items = pd.read_csv('../processed_dataset/Amazon-Beauty/amazon_beauty_reviews_meta_modified.csv')
# movies = pd.read_csv('processed_dataset/MovieLens-1M/movies/movies_movielens.csv')

full_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/10 interactions/amazon_beauty_ratings_full.csv')
train_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/10 interactions/amazon_beauty_ratings_train.csv')
val_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/10 interactions/amazon_beauty_ratings_val.csv')
test_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/10 interactions/amazon_beauty_ratings_test.csv')

# full_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_full.csv')
# train_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_train.csv')
# val_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_val.csv')
# test_ratings = pd.read_csv('../processed_dataset/Amazon-Beauty/20 interactions/amazon_beauty_ratings_test.csv')

In [67]:
train_ratings

,rating,title,text,images,asin,parent_asin,user_id,timestamp,verified_purchase,helpful_vote
0,5,Very pretty and shiny headband!,I got this for my headband-loving almost 6 yea...,[{'small_image_url': 'https://images-na.ssl-im...,B0895XPZNT,B0895XPZNT,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.615320e+12,False,0
1,5,Perfect for traveling!,These are perfect for traveling due to the sho...,[{'small_image_url': 'https://images-na.ssl-im...,B08SQXBZ5W,B08SQXBZ5W,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.618160e+12,False,1
2,5,So fun for “princess hair”!,I am impressed with the feel of this wig! I go...,[{'small_image_url': 'https://images-na.ssl-im...,B08S3VFJ22,B08S3VFJ22,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.618950e+12,False,2
3,4,"Beautiful, fun wig!",This is a super fun dress up wig! It is well m...,[{'small_image_url': 'https://images-na.ssl-im...,B0863FZFPV,B0863FZFPV,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.619050e+12,False,0
4,4,Nice headbands,These headbands are as advertised and lightwei...,[{'small_image_url': 'https://images-na.ssl-im...,B09188R9KV,B09188R9KV,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.620510e+12,False,0
...,...,...,...,...,...,...,...,...,...,...
3424,5,two for five is a great investment for my razor,Glad that extra cartridges for Bladelife razor...,[],B09B2YF8NM,B09B2YF8NM,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.648220e+12,False,0
3425,5,Love the color. Long-Lasting. Need one coat.,I like this Rose Pose liquid lip color. It is ...,[{'small_image_url': 'https://m.media-amazon.c...,B07C7S9WNP,B0BVQQ58G8,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.660290e+12,False,0
3426,3,"don't care for the smell, won't use it",Absorbs quickly and does not leave skin oily.<...,[],B07JDKTKPW,B07JDKTKPW,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.661150e+12,False,0
3427,4,Straightens hair but not sleek (flat-iron) str...,This straightener is practical and easy to use...,[],B08LKLDCWJ,B08LKLDCWJ,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.675960e+12,False,0


In [68]:
items

,main_category,title,features,description,store,details,parent_asin
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",[],[],Howard Products,{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,[],[],Yes To,"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),[],[],Levine Health Products,{'Manufacturer': 'Levine Health Products'},B000B658RI
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",[],[],Cherioll,"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",B088FKY3VD
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,"['Material: 304 Stainless Steel; Brass tip', '...",['The Precision Plunger Bars are designed to w...,Precision,{'UPC': '644287689178'},B07NGFDN6G
...,...,...,...,...,...,...,...
112585,All Beauty,"TOPREETY 24""120gr 3/4 Full Head clip in hair e...",[],[],TOPREETY,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B077D2Z5RF
112586,All Beauty,"Pets Playmate Pet Grooming Glove,Gentle Deshed...",[],[],Pets Playmate,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B07DLRYKQZ
112587,All Beauty,[10Pack] Makeup Brushes Set Cosmetics Tools Ki...,[],[],RainMakers,"{'Brand': 'RainMakers', 'Recommended Uses For ...",B07HNP2NTF
112588,All Beauty,Xcoser Pretty Party Anna Wig Hair Tails Hair S...,[],[],Xcoser,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B00U3OB8PY


In [69]:
def generate_user_texts_with_history(items, ratings):
    user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
    user_texts = []

    # Convert relevant columns to dictionaries for faster access
    items_dict = items.set_index('parent_asin')[['title', 'store']].to_dict('index')

    for _, row in ratings.iterrows():
        user_id = row['user_id']
        item_id = row['parent_asin']

        # Append the user's history (only the last 3 movies)
        history_movies = [
            f"title: {items_dict[mid]['title']} [SEP] store: {items_dict[mid]['store']}"
            for mid in user_histories[user_id][-3:]
        ]

        history_str = ", ".join(history_movies)

        # Combine user history into the final text format
        if history_str:
            combined_features = f"history: {history_str}"
        else:
            combined_features = f"history: None"

        user_texts.append(combined_features)

        # Update the user history after generating combined features
        user_histories[user_id].append(item_id)

    return user_texts

# def generate_user_texts_with_history(items, ratings):
#     user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
#     user_texts = []
#
#     # Convert relevant columns to dictionaries for faster access
#     # user_features_dict = users.set_index('user_id').to_dict('index')
#     items_dict = items.set_index('parent_asin')['genres'].to_dict()
#     # movie_titles_dict = movies.set_index('item_id')['title'].to_dict()
#
#     # movie_titles_dict = movies.set_index('item_id')[['title', 'genres']].to_dict('index')
#
#     for _, row in ratings.iterrows():
#         user_id = row['user_id']
#         movie_id = row['item_id']
#
#         # Get user features
#         user = user_features_dict[user_id]
#         user_features = f"[USER_PROFILE] occupation: {user['occupation']} [SEP] gender: {user['gender']}"
#         # user_features = f"occupation: {user['occupation']} [SEP] age: {user['age']} [SEP] gender: {user['gender']}"
#         # user_features = f"occupation: {user['occupation']} [SEP] gender: {user['gender']}"
#
#         # Append the user's history (only the last 3 movies)
#         # history_movies = [movie_titles_dict[mid] for mid in user_histories[user_id][-3:]]
#         history_movies = [movie_titles_dict[mid] for mid in user_histories[user_id][-3:]]
#
#         # history_movies = [
#         #     f"{movie_titles_dict[mid]['title']} [SEP] {movie_titles_dict[mid]['genres']}"
#         #     for mid in user_histories[user_id][-3:]
#         # ]
#
#         history_str = ", ".join(history_movies)
#
#         # Combine user features and history
#         # if history_str:
#         #     combined_features = f"{user_features} [SEP] history: {history_str}"
#         # else:
#         #     combined_features = f"{user_features}"
#
#         if history_str:
#             combined_features = f"{user_features} [SEP] genres: {history_str}"
#         else:
#             combined_features = f"{user_features}"
#
#         user_texts.append(combined_features)
#
#         # Update the user history after generating combined features
#         user_histories[user_id].append(movie_id)
#
#     return user_texts


In [70]:
train_user_texts = generate_user_texts_with_history(items, train_ratings)
val_user_texts = generate_user_texts_with_history(items, val_ratings)
test_user_texts = generate_user_texts_with_history(items, test_ratings)
train_user_texts

['history: None',
 'history: title: QTMY Fashion Rhinestone Flower Headband,Wide Hair Hoop for Women,Hair Accessories Head Band Headwear,White [SEP] store: QTMY',
 'history: title: QTMY Fashion Rhinestone Flower Headband,Wide Hair Hoop for Women,Hair Accessories Head Band Headwear,White [SEP] store: QTMY, title: Makeup Brushes with Bag, 8 Pieces Makeup Brush Set, Travel Size Cosmetic Brushes Kit for Face Foundation Blush Eye Shadow Lip and Eye brow (Pink) [SEP] store: DOYIZZ',
 'history: title: QTMY Fashion Rhinestone Flower Headband,Wide Hair Hoop for Women,Hair Accessories Head Band Headwear,White [SEP] store: QTMY, title: Makeup Brushes with Bag, 8 Pieces Makeup Brush Set, Travel Size Cosmetic Brushes Kit for Face Foundation Blush Eye Shadow Lip and Eye brow (Pink) [SEP] store: DOYIZZ, title: ENTRANCED STYLES Ombre Blonde Long Wavy Wigs for Women Blonde Wavy Wig Middle Part Heat Resistant Synthetic Wig Natural Looking Daily Party Use 26 Inch [SEP] store: ENTRANCED STYLES',
 'history

In [71]:
# def generate_last_user_texts_with_history(items, ratings):
#     user_histories = {user_id: [] for user_id in ratings['user_id'].unique()}
#     last_user_texts = {}
#
#     items_dict = items.set_index('parent_asin')[['title', 'store']].to_dict('index')
#
#     for _, row in ratings.iterrows():
#         user_id = row['user_id']
#         item_id = row['parent_asin']
#
#         # Append the user's history (only the last 3 items)
#         history_items = [
#             f"title: {items_dict[mid]['title']} [SEP] store: {items_dict[mid]['store']}"
#             for mid in user_histories[user_id][-3:]
#         ]
#
#         history_str = ", ".join(history_items)
#
#         # Combine history into the final text format
#         if history_str:
#             combined_features = f"history: {history_str}"
#         else:
#             combined_features = "history: None"
#
#         # Update the dictionary to keep the last text for each user
#         last_user_texts[user_id] = combined_features
#
#         # Update the user history after generating combined features
#         user_histories[user_id].append(item_id)
#
#     return last_user_texts
#
# # Generate the last user texts for the validation data
# val_last_user_texts = generate_last_user_texts_with_history(items, val_ratings)

def generate_last_user_texts_with_history(items, train_ratings, val_ratings):
    user_histories = {user_id: [] for user_id in train_ratings['user_id'].unique()}
    last_user_texts = {}

    items_dict = items.set_index('parent_asin')[['title', 'store']].to_dict('index')

    # First, populate the user histories using train_ratings
    for _, row in train_ratings.iterrows():
        user_id = row['user_id']
        item_id = row['parent_asin']
        user_histories[user_id].append(item_id)

    # Now process the val_ratings
    for _, row in val_ratings.iterrows():
        user_id = row['user_id']
        item_id = row['parent_asin']

        # Append the user's history (only the last 3 items)
        history_items = [
            f"title: {items_dict[mid]['title']} [SEP] store: {items_dict[mid]['store']}"
            for mid in user_histories[user_id][-3:]
        ]

        history_str = ", ".join(history_items)

        # Combine history into the final text format
        if history_str:
            combined_features = f"history: {history_str}"
        else:
            combined_features = "history: None"

        # Update the dictionary to keep the last text for each user
        last_user_texts[user_id] = combined_features

        # Update the user history after generating combined features
        user_histories[user_id].append(item_id)

    # Handle cases where history is None by using train_ratings history
    for user_id, history in last_user_texts.items():
        if history == "history: None" and user_histories[user_id]:
            history_items = [
                f"title: {items_dict[mid]['title']} [SEP] store: {items_dict[mid]['store']}"
                for mid in user_histories[user_id][-3:]
            ]
            history_str = ", ".join(history_items)
            last_user_texts[user_id] = f"history: {history_str}"

    return last_user_texts

# Generate the last user texts for the validation data
val_last_user_texts = generate_last_user_texts_with_history(items, train_ratings, val_ratings)


In [72]:
len(test_user_texts)

490

In [73]:
print(val_last_user_texts.get(5))

None


In [74]:
print(test_user_texts[2])

history: title: Mane Tame Deluxe Pomade 3.3oz/97mL - Firm Hold, Medium Shine - Made in USA, Water-Based, Combable, Rinses Out Easily, BARBER RECOMMENDED - Best used on dry hair [SEP] store: Mane Tame Professional Men's Grooming, title: Easydew EX Repair Control Renewal Moisture 1.70 fl.oz. [SEP] store: EASYDEW


In [15]:
# # Save user embeddings locally
# with open('train_user_texts.pkl', 'wb') as f:
#     pickle.dump(train_user_texts, f)
#
# print("Train user embeddings saved successfully.")
#
# with open('val_user_texts.pkl', 'wb') as f:
#     pickle.dump(val_user_texts, f)
#
# print("Validation user embeddings saved successfully.")
#
# with open('test_user_texts.pkl', 'wb') as f:
#     pickle.dump(test_user_texts, f)
#
# print("Test user embeddings saved successfully.")

In [16]:
# # Load user texts from file
# with open('./text_for_embeddings/last_three_history/train_user_texts.pkl', 'rb') as f:
#     train_user_texts = pickle.load(f)
# print("Train user text loaded successfully.")
#
# with open('./text_for_embeddings/last_three_history/val_user_texts.pkl', 'rb') as f:
#     val_user_texts = pickle.load(f)
# print("Validation user text loaded successfully.")
#
# with open('./text_for_embeddings/last_three_history/test_user_texts.pkl', 'rb') as f:
#     test_user_texts = pickle.load(f)
# print("Test user text loaded successfully.")

In [75]:
# Combine movie features into a single string for each movie
# movies['movie_features'] = 'title: ' + movies['title'] + ' [SEP] genres: ' + movies['genres']
items['store'].fillna('', inplace=True)

# Define movie_features with a condition
items['movie_features'] = items.apply(
    lambda row: f"title: {row['title']} [SEP] store: {row['store']}" if row['store'] else f"title: {row['title']} [SEP] store: None",
    axis=1
)# movies['movie_features'] = '[MOVIE_DETAIL] genres: ' + movies['genres']


C:\Users\Hooman\AppData\Local\Temp\ipykernel_19464\1666752697.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  items['store'].fillna('', inplace=True)


In [76]:
items

,main_category,title,features,description,store,details,parent_asin,movie_features
0,All Beauty,"Howard LC0008 Leather Conditioner, 8-Ounce (4-...",[],[],Howard Products,{'Package Dimensions': '7.1 x 5.5 x 3 inches; ...,B01CUPMQZE,"title: Howard LC0008 Leather Conditioner, 8-Ou..."
1,All Beauty,Yes to Tomatoes Detoxifying Charcoal Cleanser ...,[],[],Yes To,"{'Item Form': 'Powder', 'Skin Type': 'Acne Pro...",B076WQZGPM,title: Yes to Tomatoes Detoxifying Charcoal Cl...
2,All Beauty,Eye Patch Black Adult with Tie Band (6 Per Pack),[],[],Levine Health Products,{'Manufacturer': 'Levine Health Products'},B000B658RI,title: Eye Patch Black Adult with Tie Band (6 ...
3,All Beauty,"Tattoo Eyebrow Stickers, Waterproof Eyebrow, 4...",[],[],Cherioll,"{'Brand': 'Cherioll', 'Item Form': 'Powder', '...",B088FKY3VD,"title: Tattoo Eyebrow Stickers, Waterproof Eye..."
4,All Beauty,Precision Plunger Bars for Cartridge Grips – 9...,"['Material: 304 Stainless Steel; Brass tip', '...",['The Precision Plunger Bars are designed to w...,Precision,{'UPC': '644287689178'},B07NGFDN6G,title: Precision Plunger Bars for Cartridge Gr...
...,...,...,...,...,...,...,...,...
112585,All Beauty,"TOPREETY 24""120gr 3/4 Full Head clip in hair e...",[],[],TOPREETY,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B077D2Z5RF,"title: TOPREETY 24""120gr 3/4 Full Head clip in..."
112586,All Beauty,"Pets Playmate Pet Grooming Glove,Gentle Deshed...",[],[],Pets Playmate,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B07DLRYKQZ,"title: Pets Playmate Pet Grooming Glove,Gentle..."
112587,All Beauty,[10Pack] Makeup Brushes Set Cosmetics Tools Ki...,[],[],RainMakers,"{'Brand': 'RainMakers', 'Recommended Uses For ...",B07HNP2NTF,title: [10Pack] Makeup Brushes Set Cosmetics T...
112588,All Beauty,Xcoser Pretty Party Anna Wig Hair Tails Hair S...,[],[],Xcoser,"{'Is Discontinued By Manufacturer': 'No', 'Pac...",B00U3OB8PY,title: Xcoser Pretty Party Anna Wig Hair Tails...


In [77]:
items['movie_features']

0         title: Howard LC0008 Leather Conditioner, 8-Ou...
1         title: Yes to Tomatoes Detoxifying Charcoal Cl...
2         title: Eye Patch Black Adult with Tie Band (6 ...
3         title: Tattoo Eyebrow Stickers, Waterproof Eye...
4         title: Precision Plunger Bars for Cartridge Gr...
                                ...                        
112585    title: TOPREETY 24"120gr 3/4 Full Head clip in...
112586    title: Pets Playmate Pet Grooming Glove,Gentle...
112587    title: [10Pack] Makeup Brushes Set Cosmetics T...
112588    title: Xcoser Pretty Party Anna Wig Hair Tails...
112589    title: DVIO Men's Voyage Perfume, Spicy woody ...
Name: movie_features, Length: 112590, dtype: object

In [78]:
# Create a dictionary for fast lookup
item_features_dict = items.set_index('parent_asin')['movie_features'].to_dict()

# Create lists of user and item texts
item_texts = [item_features_dict[itemId] for itemId in full_ratings['parent_asin'].unique()]

# Create a mapping from userId and movieId to indices
item_id_to_idx = {itemId: idx for idx, itemId in enumerate(full_ratings['parent_asin'].unique())}

# Map userId and movieId in ratings_df to indices
train_ratings['parent_asin_idx'] = train_ratings['parent_asin'].map(item_id_to_idx)

# Map userId and movieId in ratings_val to indices
val_ratings['parent_asin_idx'] = val_ratings['parent_asin'].map(item_id_to_idx)

# Map userId and movieId in ratings_test to indices
test_ratings['parent_asin_idx'] = test_ratings['parent_asin'].map(item_id_to_idx)

# Extract user indices, item indices, and ratings
train_item_indices = torch.LongTensor(train_ratings['parent_asin_idx'].values).to(device)
train_labels = torch.FloatTensor(train_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for validation
val_item_indices = torch.LongTensor(val_ratings['parent_asin_idx'].values).to(device)
val_labels = torch.FloatTensor(val_ratings['rating'].values).to(device)

# Extract user indices, item indices, and ratings for test
test_item_indices = torch.LongTensor(test_ratings['parent_asin_idx'].values).to(device)
test_labels = torch.FloatTensor(test_ratings['rating'].values).to(device)


In [79]:
len(test_labels)

490

In [80]:
test_ratings

,rating,title,text,images,asin,parent_asin,user_id,timestamp,verified_purchase,helpful_vote,parent_asin_idx
0,4,Good hold with slight shine,This product is tacky so it has a good hold if...,[{'small_image_url': 'https://images-na.ssl-im...,B07K1QJH5P,B07K1QJH5P,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.610920e+12,False,0,1846
1,4,Pretty decent product but do your research,"Overall, this product isn’t horrible compared ...",[{'small_image_url': 'https://images-na.ssl-im...,B01N7A5AGF,B01N7A5AGF,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.613920e+12,False,2,722
2,5,Take me back to the 90s!,I had a set of headbands JUST like these in th...,[{'small_image_url': 'https://images-na.ssl-im...,B08QHP717Z,B08QHP717Z,AE3KLVXGZPANXE5XLXYKHTVAZ3FQ,1.614550e+12,False,0,230
3,4,"Nice Cleanser That Leaves No Oily Residue, But...",This Neutrogena cleansing oil is nice and does...,[],B00U2VQZC4,B00U2VQZC4,AE3PLZHW6NXWBMZ76TDVFQG2MJFA,1.441240e+12,False,0,3340
4,5,Eye Replacement Head Fits Glo Pro Perfectly & ...,I’m using the Glo Pro eye attachment head with...,[{'small_image_url': 'https://images-na.ssl-im...,B06XD3SXQ8,B09J5TZ7HL,AE3PLZHW6NXWBMZ76TDVFQG2MJFA,1.539880e+12,False,16,1589
...,...,...,...,...,...,...,...,...,...,...,...
485,5,Love the convenient handle,This does a great job exfoliating the skin and...,[{'small_image_url': 'https://images-na.ssl-im...,B08B5FJMHM,B08B5FJMHM,AHY2TURQPNIDXZGH2CMQLZ343YMQ,1.598400e+12,False,0,10905
486,4,I can use this even with my sensitive eyes and...,I have sensitive skin and eyes. Without the r...,[],B01CYTUXHO,B01CYTUXHO,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.477420e+12,False,1,3385
487,5,Big and thick with little sponge hearts!!!,These are great!! They are big and thick with ...,[],B01N2XUHTW,B01N2XUHTW,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.493010e+12,False,0,1822
488,5,Sturdy!,Great tweezers!!<br />Sturdy set!!,[],B01GL4HV64,B01GL4HV64,AHYOSWORVZFXM5QMRIAW3JTTFFIQ,1.497500e+12,False,2,689


In [81]:
from torch.utils.data import Dataset, DataLoader

class CustomTextDataset(Dataset):
    def __init__(self, users, item_ids, ratings):
        self.users = users
        self.item_ids = item_ids
        self.ratings = ratings

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        users = self.users[idx]
        item_id = self.item_ids[idx]
        rating = self.ratings[idx]
        return users, item_id, rating

In [82]:
# Create DataLoader for training data
train_dataset = CustomTextDataset(train_user_texts, train_item_indices, train_labels)
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for validation data
val_dataset = CustomTextDataset(val_user_texts, val_item_indices, val_labels)
val_dataloader = DataLoader(val_dataset, batch_size=64, shuffle=True, drop_last=True)

# Create DataLoader for test data
test_dataset = CustomTextDataset(test_user_texts, test_item_indices, test_labels)
test_dataloader = DataLoader(test_dataset, batch_size=64, shuffle=True, drop_last=True)

In [83]:
class TwoTowerModel(pl.LightningModule):
    def __init__(self, user_model_name, item_model_name, embedding_size=384):
        super(TwoTowerModel, self).__init__()
        self.user_model = SentenceTransformer(user_model_name)
        self.item_model = SentenceTransformer(item_model_name)

        self.user_fc = nn.Linear(embedding_size, embedding_size)
        self.item_fc = nn.Linear(embedding_size, embedding_size)

        self.criterion = nn.MSELoss()
        self.epoch_losses = {'train_loss': [], 'val_loss': []}

    def forward(self, user_text, item_text):
        user_embedding = self.user_model.encode(user_text, convert_to_tensor=True).to(device)
        item_embedding = self.item_model.encode(item_text, convert_to_tensor=True).to(device)

        user_output = self.user_fc(user_embedding)
        item_output = self.item_fc(item_embedding)

        dot_product = torch.matmul(user_output.squeeze(), item_output.T)
        dot_product = 4 * torch.sigmoid(dot_product) + 1

        return dot_product

    def training_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        users, items, ratings = batch

        items = [item_texts[i] for i in items.tolist()]

        preds = self(users, items)

        loss = self.criterion(preds, ratings)
        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=1e-5)

class PrintLossesCallback(Callback):
    def on_train_epoch_end(self, trainer, pl_module):
        train_loss = trainer.callback_metrics.get('train_loss')
        if train_loss is not None:
            pl_module.epoch_losses['train_loss'].append(train_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Train Loss: {train_loss.item()}")

    def on_validation_epoch_end(self, trainer, pl_module):
        val_loss = trainer.callback_metrics.get('val_loss')
        if val_loss is not None:
            pl_module.epoch_losses['val_loss'].append(val_loss.item())
            print(f"Epoch {trainer.current_epoch + 1}: Val Loss: {val_loss.item()}")

In [84]:
# model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2')
model = TwoTowerModel(user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2')

# Define the ModelCheckpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',  # Metric to monitor
    dirpath='checkpoints/',  # Directory to save the checkpoints
    filename='with-history-best-checkpoint',  # Filename for the best model
    save_top_k=1,  # Save only the top 1 model
    mode='min'  # Mode to save the best model (min for validation loss)
)

trainer = pl.Trainer(max_epochs=10, log_every_n_steps=1, callbacks=[PrintLossesCallback()], enable_progress_bar=True)
trainer.fit(model, train_dataloader, val_dataloader)

# Print losses after training completes
print("Epoch losses:")
for epoch in range(trainer.max_epochs):
    train_loss = model.epoch_losses['train_loss'][epoch] if epoch < len(model.epoch_losses['train_loss']) else 'N/A'
    val_loss = model.epoch_losses['val_loss'][epoch] if epoch < len(model.epoch_losses['val_loss']) else 'N/A'
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss}, Val Loss: {val_loss}")

D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name       | Type                | Params | Mode 
-----------------------------------------------------------
0 | user_model | SentenceTransformer | 33.4 M | train
1 | item_model | SentenceTransformer | 33.4 M | train
2 | user_fc    | Linear              | 147 K  | train
3 | item_fc    | Linear              | 147 K  | train
4 | criterion  | MSELoss             | 0      | train
-----------------------------------------------------------
67.0 M    Trainable params
0         Non-trainable params
67.0 M    Total params
268.063   Total estimated model params siz

Sanity Checking DataLoader 0:   0%|          | 0/2 [00:00<?, ?it/s]

D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:475: Your `val_dataloader`'s sampler has shuffling enabled, it is strongly recommended that you turn shuffling off for val/test dataloaders.
D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Sanity Checking DataLoader 0: 100%|██████████| 2/2 [00:00<00:00,  3.34it/s]Epoch 1: Val Loss: 2.334233522415161
                                                                           

D:\Anaconda\lib\site-packages\torch\nn\modules\loss.py:535: UserWarning: Using a target size (torch.Size([64])) that is different to the input size (torch.Size([64, 64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


D:\Anaconda\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Epoch 0: 100%|██████████| 53/53 [00:13<00:00,  3.81it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 6/6 [00:01<00:00,  5.65it/s]Epoch 1: Val Loss: 1.400038719177246

Epoch 1: 100%|██████████| 53/53 [00:13<00:00,  4.06it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 6/6 [00:01<00:00,  5.15it/s]Epoch 2: Val Loss: 1.1237744092941284

Epoch 2: 100%|██████████| 53/53 [00:13<00:00,  4.01it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 6/6 [00:01<00:00,  5.53it/s]Epoch 3: Val Loss: 1.0230700969696045

Epoch 3: 100%|██████████| 53/53 [00:13<00:00,  3.92it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Validation DataLoader 0: 100%|██████████| 6/6 [00:01<00:00,  5.60it/s]Epoch 4: Val Loss: 0.9762786030769348

Epoch 4: 100%|██████████| 53/53 [00:12<00:00,  4.18it/s, v_num=0]
Validation: |          | 0/? [00:00<?, ?it/s]
Valid

`Trainer.fit` stopped: `max_epochs=10` reached.


Epoch 9: 100%|██████████| 53/53 [00:24<00:00,  2.19it/s, v_num=0]
Epoch losses:
Epoch 1: Train Loss: 1.3161587715148926, Val Loss: 2.334233522415161
Epoch 2: Train Loss: 0.9139474630355835, Val Loss: 1.400038719177246
Epoch 3: Train Loss: 0.793451189994812, Val Loss: 1.1237744092941284
Epoch 4: Train Loss: 1.5407934188842773, Val Loss: 1.0230700969696045
Epoch 5: Train Loss: 0.9534710049629211, Val Loss: 0.9762786030769348
Epoch 6: Train Loss: 0.777935266494751, Val Loss: 0.9712547659873962
Epoch 7: Train Loss: 1.1352016925811768, Val Loss: 0.8713609576225281
Epoch 8: Train Loss: 1.0606399774551392, Val Loss: 0.9634180068969727
Epoch 9: Train Loss: 0.7057255506515503, Val Loss: 0.8138418197631836
Epoch 10: Train Loss: 1.2080790996551514, Val Loss: 0.9650018811225891


In [20]:
model.epoch_losses

{'train_loss': [0.9391066431999207,
  1.0028347969055176,
  0.9369568824768066,
  0.9828064441680908,
  0.9409568905830383],
 'val_loss': [10.464865684509277,
  1.1707780361175537,
  1.138741135597229,
  1.120898962020874,
  1.1324385404586792,
  1.116618275642395]}

# Evaluation

In [85]:
# Assuming the training part has been done already, load the best model checkpoint


# best_model_path = './lightning_logs/paraphrase-MiniLM-L12-v2/not-binarized/user_title + store & item_ title + store  _ 5 epoch/checkpoints/epoch=4-step=435.ckpt'
best_model_path = './lightning_logs/version_0/checkpoints/epoch=9-step=530.ckpt'

# best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L6-v2', item_model_name='paraphrase-MiniLM-L6-v2').to(device)
best_model = TwoTowerModel.load_from_checkpoint(best_model_path, user_model_name='paraphrase-MiniLM-L12-v2', item_model_name='paraphrase-MiniLM-L12-v2').to(device)


D:\Anaconda\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


## Calculations

In [86]:
def get_top_n_items_without_history_unseen_items(model, userId, n):
    # Ensure the model is in evaluation mode
    model.eval()

    # Get the user text for the given userId
    user_text = val_last_user_texts[userId]
    # print(user_text)
    # Encode the user text
    user_embedding = model.user_model.encode(user_text, convert_to_tensor=True).to(device)

    # Compute the scores (dot product between user embedding and each item embedding)
    user_output = model.user_fc(user_embedding).to(device)
    item_output = model.item_fc(full_items_embeddings).to(device)
    dot_product = torch.matmul(user_output, item_output.t()).squeeze()

    # Get items the user has seen in the training and validation data
    seen_items_train = train_ratings[train_ratings['user_id'] == userId]['parent_asin'].values
    seen_items_val = val_ratings[val_ratings['user_id'] == userId]['parent_asin'].values
    seen_items = set(np.concatenate((seen_items_train, seen_items_val)))
    # print(len(seen_items))
    # Get the top n + len(seen_items) item indices and their scores
    # top_n_scores, top_n_indices = torch.topk(dot_product, n + len(seen_items))
    top_n_scores, top_n_indices = torch.topk(dot_product, n)

    # Map indices back to item IDs
    top_n_item_ids = [list(item_id_to_idx.keys())[list(item_id_to_idx.values()).index(idx.item())] for idx in top_n_indices]

    # Filter out seen items
    # unseen_top_n_item_ids = [item for item in top_n_item_ids if item not in seen_items]
    # print(unseen_top_n_item_ids[:n])
    # return unseen_top_n_item_ids[:n]
    # print(top_n_item_ids[:n])
    return top_n_item_ids[:n]


In [87]:
item_texts[27]

'title: Charcoal Konjac Face Sponge 3 pk | Acne, Psoriasis, Bumpy Skin & Ingrown Hairs [SEP] store: None'

In [88]:
# Assuming full_items_embeddings is already defined
full_items_embeddings = torch.stack([best_model.item_model.encode(item_text, convert_to_tensor=True) for item_text in item_texts[27]]).to(device)

In [89]:
item_texts[8]

'title: NIRA Skincare Laser & Serum Bundle - Includes Anti-Aging Laser & Hyaluronic Acid Serum - Reduces Appearance of Fine Lines & Wrinkles - FDA Cleared [SEP] store: Nira'

## Type 0

In [94]:
def dcg(scores, k):
    scores = np.asfarray(scores)[:k]
    return np.sum(scores / np.log2(np.arange(2, scores.size + 2)))

def ndcg_at_k(labels, k):
    ideal_labels = sorted(labels, reverse=True)
    return dcg(labels, k) / dcg(ideal_labels, k)

def recall_at_k(labels, relevant_count, k):
    return np.sum(labels[:k]) / relevant_count

def mrr_at_k(labels, k):
    for i, label in enumerate(labels[:k]):
        if label == 1:
            return 1 / (i + 1)
    return 0

def evaluate_user_cf_model(model, test_data, k, item_titles):
    ndcg_scores = []
    recall_scores = []
    mrr_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # print(user)
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)
        # recommended_titles = [item_titles.get(item, "Unknown Title") for item in recommended_items]

        # print("Recommended items and their titles:")

        # for item, title in zip(recommended_items, recommended_titles):
        #     print(f"{item}: {title}")

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['parent_asin'].values

        y_score = [1 if item in test_items else 0 for item in recommended_items]
        # print(y_score)
        ndcg = ndcg_at_k(y_score, k)
        recall = recall_at_k(y_score, len(test_items), k)
        mrr = mrr_at_k(y_score, k)

        ndcg_scores.append(ndcg)
        recall_scores.append(recall)
        mrr_scores.append(mrr)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))

    avg_ndcg = np.nanmean(ndcg_scores)
    avg_recall = np.nanmean(recall_scores)
    avg_mrr = np.nanmean(mrr_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg,
        'Recall@{}'.format(k): avg_recall,
        'MRR@{}'.format(k): avg_mrr,
    }


all_items = items['parent_asin'].unique()
item_titles = dict(zip(items['parent_asin'], items['title']))
# Evaluate the model
# eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5, item_titles=item_titles)
# print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=100, item_titles=item_titles)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_19464\261760093.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)


{'NDCG@100': 0.17949890370636482, 'Recall@100': 0.053827399905831276, 'MRR@100': 0.00428180832307471}


## Type 2

In [91]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['parent_asin'].values

        y_score = [
            user_test_data[user_test_data['parent_asin'] == item]['rating'].values[0] if item in test_items else 2.5
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['parent_asin'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=10)
print(eval_result)

{'NDCG@5': 1.0}
{'NDCG@10': 1.0}


## Type 3

In [92]:
def evaluate_user_cf_model(model, test_data, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['parent_asin'].values
        # print(user)

        y_score = [
            user_test_data[user_test_data['parent_asin'] == item]['rating'].values[0] if item in test_items else 0
            for item in recommended_items
        ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = items['parent_asin'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, k=10)
print(eval_result)

C:\Users\Hooman\AppData\Local\Temp\ipykernel_19464\1899615334.py:7: RuntimeWarning: invalid value encountered in scalar divide
  return dcg(labels, k) / dcg(ideal_labels, k)
C:\Users\Hooman\AppData\Local\Temp\ipykernel_19464\1072997075.py:23: RuntimeWarning: Mean of empty slice
  avg_ndcg = np.nanmean(ndcg_scores)


{'NDCG@5': nan}
{'NDCG@10': nan}


## Type 1

In [ ]:
def evaluate_user_cf_model(model, test_data, train_data, val_data, all_items, k):
    ndcg_scores = []

    # Get unique users
    unique_users = test_data['user_id'].unique()

    for user in unique_users:
        # Get the top N items for the user, filtering out seen items
        recommended_items = get_top_n_items_without_history_unseen_items(model, user, k)

        user_test_data = test_data[test_data['user_id'] == user]
        test_items = user_test_data['item_id'].values
        print(user)
        # y_score = [
        #     user_test_data[user_test_data['item_id'] == item]['rating'].values[0] if item in test_items else 0
        #     for item in recommended_items
        # ]
        y_score = [
            1 if (item in test_items and user_test_data[user_test_data['item_id'] == item]['label'].values[0] == 1) else 0
            for item in recommended_items
        ]
        # y_score = [
        #     1 if (item in test_items and user_test_data[user_test_data['item'] == item]['label'].values[0] == 1) else 0
        #     for item in recommended_items
        # ]

        ndcg = ndcg_at_k(y_score, k)
        ndcg_scores.append(ndcg)

    # avg_ndcg = np.mean(np.nan_to_num(ndcg_scores, nan=0.0))
    avg_ndcg = np.nanmean(ndcg_scores)

    return {
        'NDCG@{}'.format(k): avg_ndcg
    }

all_items = movies['item_id'].unique()
# Evaluate the model
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)
eval_result = evaluate_user_cf_model(best_model, test_ratings, train_ratings, val_ratings, all_items, k=5)
print(eval_result)